In [ ]:
Tok=""

In [10]:
import pandas as pd
from transformers import AutoTokenizer,AutoModelForCausalLM,Trainer,TrainingArguments, pipeline
from datasets import Dataset

In [7]:
df=pd.read_csv('set.csv')
df=df.dropna(axis=1)
df.head(3)
df=df[['empathetic_dialogues','labels']]
df=df.head(10)
df

,empathetic_dialogues,labels
0,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju..."
1,Customer :This was a best friend. I miss her.\...,Where has she gone?
2,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...
3,Customer :Was this a friend you were in love w...,This was a best friend. I miss her.
4,Customer :Where has she gone?\nAgent :,We no longer talk.
5,Customer : it feels like hitting to blank wall...,Oh ya? I don't really see how
6,Customer :dont you feel so.. its a wonder \nAg...,I do actually hit blank walls a lot of times b...
7,Customer : i virtually thought so.. and i used...,Wait what are sweatings
8,Customer :Oh ya? I don't really see how\nAgent :,dont you feel so.. its a wonder
9,Customer :I do actually hit blank walls a lot ...,i virtually thought so.. and i used to get sw...


In [14]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2",token=Tok)
model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")
df['text']="User: "+df["empathetic_dialogues"]+"\nAssistant: "+df["labels"]
ds=Dataset.from_pandas(df[["text"]])
tokenizer.pad_token=tokenizer.eos_token
def tok(s):
    a=tokenizer(s["text"],truncation=True,padding="max_length",max_length=128)
    a["labels"]=a["input_ids"].copy()
    return a
ds=ds.map(tok)
args=TrainingArguments(output_dir="model",num_train_epochs=3,per_device_train_batch_size=2,save_strategy="no")
trainer=Trainer(model=model,args=args,train_dataset=ds)
trainer.train()
model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

Map: 100%|██████████| 10/10 [00:00<00:00, 103.01 examples/s]
c:\Users\Hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.92s/it]


('./model\\tokenizer_config.json', './model\\tokenizer.json')

In [24]:
b=pipeline("text-generation",model="model",tokenizer="model",max_new_tokens=80)
while True:
    usr = input()
    print(f"USER: {usr}")
    if usr.lower() == "quit":
        break
    p=f"\nUser: {usr}\nAssistant:"
    res=b(p,max_new_tokens=80,pad_token_id=tokenizer.eos_token_id)
    ans=res[0]["generated_text"]
    ans=ans.split("Assistant:")[-1].strip()
    print("\nBOT:",ans)


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2405.54it/s]
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER: hello

BOT: what is your job?


[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


USER: i am feeling depressed

BOT: it is hard for me to feel like it is like a man and this is not just the case.
USER: quit
